# Whisper Transcription Bot

Goal: start with one short YouTube talk, then scale the same workflow to 10 short NLP conference talks.

This notebook uses **Whisper** for ASR. The homework wording says "Transcribe with Tesseract", but Tesseract is OCR for text in images/screenshots; it is not the right tool for speech-to-text audio transcription.

## 1. Environment check

Run this notebook with the `llm_course_env` kernel. The local machine already has Whisper and `ffmpeg`; `yt-dlp` is available as a command-line tool.

In [1]:
import importlib.util
import shutil
import sys

print("Python:", sys.executable)
for package in ["whisper", "faster_whisper", "torch"]:
    spec = importlib.util.find_spec(package)
    print(f"{package:15s}", "OK" if spec else "missing", spec.origin if spec else "")

for command in ["yt-dlp", "ffmpeg"]:
    path = shutil.which(command)
    print(f"{command:15s}", path or "missing")

Python: /opt/homebrew/Caskroom/miniconda/base/envs/llm_course_env/bin/python
whisper         OK /opt/homebrew/Caskroom/miniconda/base/envs/llm_course_env/lib/python3.12/site-packages/whisper/__init__.py
faster_whisper  OK /opt/homebrew/Caskroom/miniconda/base/envs/llm_course_env/lib/python3.12/site-packages/faster_whisper/__init__.py
torch           OK /opt/homebrew/Caskroom/miniconda/base/envs/llm_course_env/lib/python3.12/site-packages/torch/__init__.py
yt-dlp          /opt/homebrew/bin/yt-dlp
ffmpeg          /opt/homebrew/bin/ffmpeg


## 2. Configuration

Start with `MAX_TALKS = 1` for a pilot run. After it works, add nine more URLs and set `MAX_TALKS = 10`.

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
OUTPUT_DIR = PROJECT_DIR / "hw_output" / "whisper_transcription"
AUDIO_DIR = OUTPUT_DIR / "audio"
TRANSCRIPTS_PATH = OUTPUT_DIR / "talks_transcripts.jsonl"
SCRIPT_PATH = OUTPUT_DIR / "transcribe_youtube_talks.py"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

# Replace this sample with your preferred short NLP conference talk URL.
# Tip: choose URLs where the relevant talk is already short, or use start/end seconds below.
TALKS = [
    {
        "talk_id": "pilot_001",
        "title": "Pilot NLP conference talk",
        "url": "https://www.youtube.com/watch?v=PFmVF93_f54",
        "start_seconds": None,
        "end_seconds": None,
    },
]

MAX_TALKS = 1
WHISPER_MODEL = "base"  # tiny is faster; base is still practical for ~3-minute talks.
LANGUAGE = "en"

## 3. Download and transcription helpers

In [3]:
import json
import subprocess
from typing import Any

import whisper


def run_command(command: list[str]) -> None:
    """Run a command and show useful stderr if it fails."""
    result = subprocess.run(command, text=True, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(
            "Command failed:\n"
            + " ".join(command)
            + "\n\nSTDOUT:\n"
            + result.stdout
            + "\nSTDERR:\n"
            + result.stderr
        )


def download_audio(talk: dict[str, Any], audio_dir: Path) -> Path:
    """Download YouTube audio as m4a with yt-dlp."""
    output_template = audio_dir / f"{talk['talk_id']}.%(ext)s"
    command = [
        "yt-dlp",
        "--no-playlist",
        "--extract-audio",
        "--audio-format",
        "m4a",
        "--output",
        str(output_template),
        talk["url"],
    ]
    run_command(command)

    matches = sorted(audio_dir.glob(f"{talk['talk_id']}.*"))
    if not matches:
        raise FileNotFoundError(f"No audio file was created for {talk['talk_id']}")
    return matches[0]


def clip_audio_if_needed(talk: dict[str, Any], audio_path: Path, audio_dir: Path) -> Path:
    """Optionally trim a longer video to the short talk segment."""
    start = talk.get("start_seconds")
    end = talk.get("end_seconds")
    if start is None and end is None:
        return audio_path

    clipped_path = audio_dir / f"{talk['talk_id']}_clip.wav"
    command = ["ffmpeg", "-y"]
    if start is not None:
        command += ["-ss", str(start)]
    command += ["-i", str(audio_path)]
    if end is not None and start is not None:
        command += ["-t", str(end - start)]
    elif end is not None:
        command += ["-to", str(end)]
    command += ["-ac", "1", "-ar", "16000", str(clipped_path)]
    run_command(command)
    return clipped_path


def transcribe_audio(model: Any, talk: dict[str, Any], audio_path: Path) -> dict[str, Any]:
    result = model.transcribe(str(audio_path), language=LANGUAGE, verbose=False)
    segments = [
        {
            "id": int(segment["id"]),
            "start": float(segment["start"]),
            "end": float(segment["end"]),
            "text": segment["text"].strip(),
        }
        for segment in result.get("segments", [])
    ]
    return {
        "talk_id": talk["talk_id"],
        "title": talk.get("title"),
        "url": talk["url"],
        "audio_path": str(audio_path),
        "language": result.get("language"),
        "text": result.get("text", "").strip(),
        "segments": segments,
    }


def write_jsonl(records: list[dict[str, Any]], path: Path) -> None:
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

## 4. Pilot run

Set a real YouTube URL before running this cell. It will write `hw_output/whisper_transcription/talks_transcripts.jsonl`.

In [4]:
selected_talks = TALKS[:MAX_TALKS]
placeholder_urls = [talk for talk in selected_talks if "REPLACE_WITH_YOUTUBE_ID" in talk["url"]]
if placeholder_urls:
    raise ValueError("Replace the sample YouTube URL in TALKS before running the pilot.")

model = whisper.load_model(WHISPER_MODEL)
records = []

for talk in selected_talks:
    print(f"Downloading {talk['talk_id']}: {talk.get('title')}")
    raw_audio_path = download_audio(talk, AUDIO_DIR)
    audio_path = clip_audio_if_needed(talk, raw_audio_path, AUDIO_DIR)

    print(f"Transcribing {audio_path.name}")
    record = transcribe_audio(model, talk, audio_path)
    records.append(record)

write_jsonl(records, TRANSCRIPTS_PATH)
print(f"Wrote {len(records)} transcript(s) to {TRANSCRIPTS_PATH}")

100%|███████████████████████████████████████| 139M/139M [00:31<00:00, 4.66MiB/s]


Transcribing pilot_001.m4a


/opt/homebrew/Caskroom/miniconda/base/envs/llm_course_env/lib/python3.12/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
100%|██████████| 36802/36802 [00:09<00:00, 4012.16frames/s]

Wrote 1 transcript(s) to /Users/ming/Desktop/Learning/MLE_in_Gen_AI-Course/class2/hw_output/whisper_transcription/talks_transcripts.jsonl


## 5. Preview JSONL output

In [5]:
if TRANSCRIPTS_PATH.exists():
    with TRANSCRIPTS_PATH.open("r", encoding="utf-8") as f:
        first_record = json.loads(next(f))
    print(first_record["talk_id"], first_record["title"])
    print(first_record["text"][:500])
    print("segments:", len(first_record["segments"]))
    print(first_record["segments"][:3])
else:
    print("No transcript file yet. Run the pilot cell first.")

pilot_001 Pilot NLP conference talk
I am delighted to be joined by Sarah Fletcher today. Sarah, thank you so much for joining me. Thank you. You are presenting at the NLP conference in 2024, the virtual part of the conference, and your presentation is using NLP to transform your business from the inside out. Absolutely. I think we could all do with. So please do start by telling us a little bit more about yourself and your presentation. Oh, thank you, Karen. So yes, so as you already know, I am an NLP trainer and absolutely love t
segments: 57
[{'id': 0, 'start': 0.0, 'end': 10.88, 'text': 'I am delighted to be joined by Sarah Fletcher today. Sarah, thank you so much for joining me.'}, {'id': 1, 'start': 10.88, 'end': 16.240000000000002, 'text': 'Thank you. You are presenting at the NLP conference in 2024,'}, {'id': 2, 'start': 16.240000000000002, 'end': 24.72, 'text': 'the virtual part of the conference, and your presentation is using NLP to transform your business'}]
